# Person 1 — Quant Research
Run all cells now on the offline synthetic demo. For the event, edit `config.json` first.

Own the target interpretation, availability of inputs, signal hypotheses and feature recipe.
Share the recipe with Person 2; their CV runner fits it separately in each fold.
Use the first fold's **training rows** for supervised discovery. After choosing features,
aggregate CV is a selection score; preserve a separate final holdout if time allows.

## Challenge update: forecast the full final 199 days

The team reports five-minute prices for 386 stocks in six countries. Forecast prices
throughout the final 199 days from information available before that period; do not
assume later true prices will be revealed. The exact metric and file schema are pending.
The demos below remain generic and do not implement that competition protocol automatically.

Use `qrtkit.forecasting.forecast_prices` for whole-period per-stock persistence or
training-fitted damped drift, and `qrtkit.validation.last_period_holdout` for complete
final-day validation. The [challenge notes](../docs/CHALLENGE_NOTES.md) contain an
adaptable example. Keep future actual prices out of every predictor; use frozen
history for the actual challenge, rather than the optional online financial demo.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "qrtkit").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from qrtkit import load_config, load_data, train_cv
cfg = load_config(ROOT / "config.json")
import matplotlib.pyplot as plt
from copy import deepcopy
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits
from qrtkit.features import FeatureBuilder
from qrtkit.validation import make_splits
from qrtkit.research import (eda, correlations, signal_table, feature_clusters,
                             signal_bins, information_coefficient, drift_report, lead_lag)
from qrtkit.models import FactorFeatures

data = load_data(cfg)
splits = make_splits(data.train, data.y, cfg)
discovery_idx, check_idx = splits[0]
discovery = data.train.iloc[discovery_idx]
y_discovery = data.y.iloc[discovery_idx]
print('Raw training/test shape:', data.train.shape, data.test.shape)
print('Discovery/check rows:', len(discovery_idx), len(check_idx))
display(data.train.head())

## Inspect the schema and data quality
Check whether IDs, repeated entities and dates are meaningful; verify the label horizon.
Anonymized date names must not be treated as a chronological sequence. Quantiles below
use a bounded sample for speed. Compare train/test covariates without test labels.

In [ ]:
report = eda(discovery, y_discovery, task=cfg['task'])
print({k: report[k] for k in ['shape', 'memory_mb', 'duplicate_rows']})
display(report['profile'])
display(report['numeric_summary_sample'])
display(report['target'])
display(drift_report(data.train, data.test).head(30))

## Define the reusable feature recipe
Edit `cfg['features']` below or the shared JSON. No column names are inferred.

- `histories`: lists of known history columns, **newest first**, grouped under a signal name.
- `temporal`: entity columns, numeric/datetime time, strictly past lags/windows; requires temporal CV.
- `cross_sectional`: same-time ranks/relative values; requires full available batches and whole-section folds.
- `group_statistics`: covariate means/std/counts fitted on training rows only.
- `interactions`: pairs of raw numeric columns; `drop_raw_columns` removes noisy raw histories afterward.

Example to adapt: `cfg['features']['histories'] = {'history': ['latest', 'earlier', 'oldest']}`.
Leave disabled blocks empty until there is a reason to enable them.

In [ ]:
# Make your challenge-specific edits to cfg here.
# cfg['features']['windows'] = [2, 3]
# cfg['features']['histories'] = {'history': ['latest', 'earlier', 'oldest']}

builder = FeatureBuilder(cfg)
X_discovery = builder.fit_transform(discovery)
X_check = builder.transform(data.train.iloc[check_idx])
print('Engineered feature dimensions:', X_discovery.shape, X_check.shape)
display(X_discovery.head())

## Correlations, mutual information and redundancy
These are exploratory measurements on discovery rows. MI is estimated on at most 5,000
rows; correlated features and repeated searches can exaggerate apparent evidence.
Binary label correlations use the encoded class index. Multiclass ordinal correlations
are intentionally omitted. Use held-out model performance to validate any selection.

In [ ]:
signals = signal_table(X_discovery, y_discovery, task=cfg['task'], seed=cfg['seed'])
display(signals.head(25))
corr = correlations(X_discovery, seed=cfg['seed'])
display(corr['spearman'].iloc[:12, :12])
display(feature_clusters(X_discovery).rename_axis('feature').to_frame())
if len(corr['spearman']):
    fig, ax = plt.subplots(figsize=(7, 5))
    c = corr['spearman'].iloc[:12, :12]
    im = ax.imshow(c.to_numpy(), vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_xticks(range(len(c)), c.columns, rotation=90)
    ax.set_yticks(range(len(c)), c.index)
    fig.colorbar(im, ax=ax, label='Spearman correlation')
    fig.tight_layout()
    plt.show()

In [ ]:
y_signal = (LabelEncoder().fit_transform(y_discovery) if cfg['task'] == 'classification'
            else y_discovery.to_numpy())
if len(signals) and (cfg['task'] == 'regression' or len(np.unique(y_signal)) == 2):
    best = signals.index[0]
    numeric_target = pd.Series(y_signal, index=discovery.index)
    display(signal_bins(X_discovery[best], numeric_target))
    time_column = cfg['data'].get('time_column')
    if time_column:
        ic = information_coefficient(X_discovery[best], numeric_target, discovery[time_column])
        display({k: v for k, v in ic.items() if k != 'per_group'})
        display(ic['per_group'].head())

## PCA factors and observation clusters
This discovery fit is for interpretation. Person 2 enables the same operations using
`preprocessing.pca_components` and `preprocessing.clusters`, which refits them in each fold.
Correlation clusters above group redundant **features**; KMeans here groups **observations**.

In [ ]:
numeric = X_discovery.select_dtypes(include='number').replace([np.inf, -np.inf], np.nan)
if len(numeric.columns) >= 2 and len(numeric) >= 4:
    count = min(3, len(numeric.columns), len(numeric))
    factors = Pipeline([
        ('impute', SimpleImputer(strategy='median', keep_empty_features=True)),
        ('scale', StandardScaler()),
        ('factors', FactorFeatures(pca_components=count, clusters=2, seed=cfg['seed']))
    ])
    with threadpool_limits(limits=cfg['threads']):
        factor_matrix = factors.fit_transform(numeric)
    fitted = factors.named_steps['factors']
    display(pd.DataFrame(fitted.pca_.components_.T, index=numeric.columns,
                         columns=[f'factor_{i+1}' for i in range(count)]))
    print('Explained variance:', fitted.pca_.explained_variance_ratio_)
    print('Original + PCA scores + cluster distances:', factor_matrix.shape)

## Linear baseline using the exact shared interface
Compare a simple model before spending time on feature interactions or tuning.
The full pipeline receives raw frames and refits every learned step in each fold.

In [ ]:
linear_name = 'logistic' if cfg['task'] == 'classification' else 'ridge'
linear = train_cv(data.train, data.y, data.test, cfg, linear_name)
print('OOF:', linear.metric.name, linear.score, 'coverage:', linear.covered.mean())
print('Classes:', linear.classes, 'run:', linear.output_path)

## Optional financial panel example
This independent synthetic example demonstrates temporal lags, row histories,
cross-sectional features, group mappings, and lead-lag diagnostics. Use these only
when the real challenge has suitable verified time and prediction-time availability.
It does not change your main feature recipe.

In [ ]:
from qrtkit.synthetic import make_demo, financial_config
from qrtkit.finance import fit_pair, backtest_signal
financial_cfg = financial_config(cfg)
financial_cfg['features']['windows'] = [2, 3]
financial = make_demo('financial', seed=cfg['seed'])
financial_features = FeatureBuilder(financial_cfg).fit_transform(financial.train)
display(financial_features.filter(regex='lag_|roll_|cs_|group_').head())
wide = financial.train.pivot(index='session', columns='instrument', values='move').sort_index()
display(lead_lag(wide, columns=list(wide.columns[:2]), lags=[1, 2]).head(8))

# Train-only pair hedge; a subsequent time interval is used for diagnostics.
prices = 100 * (1 + wide.iloc[:, :2]).cumprod()
cut = int(len(prices) * .7)
pair = fit_pair(prices.iloc[:cut], cointegration=False)
display(pair.transform(prices.iloc[cut:]).head())
print('Training hedge ratio:', pair.hedge_ratio)
# At session t, move[t] is observed; move[t+1] is the forward realized return.
observed = wide.iloc[:, 0]
forward = observed.shift(-1).dropna()
backtest = backtest_signal(-np.sign(observed.loc[forward.index]), forward, cost_bps=1.)
print({k: v for k, v in backtest.items() if k not in ['pnl', 'equity']})

## Hand off a recipe
Person 2 sets `RECIPE_PATH = ROOT / 'outputs/feature_recipe.json'` in their notebook,
or you merge the agreed feature/preprocessing sections into the shared `config.json`.
Only raw-data transformations and config are shared. Keep your hypothesis and any
feature-selection decisions in a short note; scores after selection are exploratory.

In [ ]:
handoff = Path(cfg['output_dir']) / 'feature_recipe.json'
handoff.parent.mkdir(parents=True, exist_ok=True)
handoff.write_text(json.dumps(cfg, indent=2))
print('Recipe ready for Person 2:', handoff)
print('Keep final validation separate from discovery; do not export full-data-fitted matrices.')